Expected Threat (xT) assigns a value to a pitch location according to the probability of eventually scoring from a possession that starts in that location, under the actions and transitions represented in the model [@singh2018xt]. Unlike expected goals (xG), which evaluates a shot, xT values positions before a shot occurs. A location far from goal can therefore have positive value if possession can progress towards more promising locations.

The xT surface is a **location-based** estimate: it summarizes typical outcomes in the event data rather than the particular players, defensive pressure, or available passing options in an individual tracking frame. PRESS uses the fitted xT grid as a spatial input, not as a substitute for its state-dependent action models.

## Mathematical formulation

Divide the pitch into spatial bins indexed by $i$. In the project formulation, a possession in bin $i$ can result in a shot, a successful pass, a successful carry, or a loss of possession. The value satisfies

$$
\mathrm{xT}_i = p_{\mathrm{shot},i}p_{\mathrm{goal},i} + p_{\mathrm{pass},i}\sum_j T^{\mathrm{pass}}_{ij}\mathrm{xT}_j + p_{\mathrm{carry},i}\sum_j T^{\mathrm{carry}}_{ij}\mathrm{xT}_j.
$$

Here $p_{\mathrm{shot},i}$ is the probability of a shot and $p_{\mathrm{goal},i}$ is its conditional scoring probability. The coefficients $p_{\mathrm{pass},i}$ and $p_{\mathrm{carry},i}$ include the probability of **successful continuation** through the respective action; they are not simply the probabilities of choosing pass or carry in Action Selection. $T^{\mathrm{pass}}_{ij}$ and $T^{\mathrm{carry}}_{ij}$ describe destination bins conditional on a successful continuation. Possession loss contributes zero future scoring value, so the displayed branch probabilities need not sum to one.

The original xT formulation groups successful ball progression into a move branch. Separating pass and carry, with an explicit zero-value loss outcome, is the project-specific formulation described here [@singh2018xt]. Starting from zero values, the equation can be iterated until the grid converges.

## Estimation from event data

The probabilities in the Bellman equation are estimated from event data, rather than learned as a single supervised prediction target. Events are assigned to pitch bins; shot outcomes, successful continuations, and their destinations supply the local probabilities and transition distributions. The fitted value surface then follows from the Bellman iteration.

**Implementation placeholder — Codex:** Insert the actual training procedure from `skillcorner-analysis` / `xt_library.py`. Document the input event selection, coordinate normalization, operational definitions of pass/carry/loss, probability denominators, smoothing, treatment of empty bins, transition-matrix normalization, convergence criterion, and the exact command that writes `results/xt/xt_grid.npy`. Verify these details against the code rather than inferring them.

## Spatial discretization and binning

Binning determines the spatial resolution of the fitted value function. Smaller bins preserve more local variation but provide fewer events per cell and sparser transition estimates; larger bins stabilize estimates but can obscure meaningful spatial structure. Smoothing can reduce sampling noise, at the cost of spatial detail.

**Binning justification placeholder — Codex:** Report the **actual** $(n_x,n_y)$ grid resolution and bin dimensions used in training. Explain how that choice was made, using available event counts/coverage, sparsity diagnostics, sensitivity analyses, or other evidence actually produced by the pipeline. If no comparison of bin sizes was performed, state the practical rationale and limitation without claiming empirical optimization. Include a compact diagnostic plot or table only if supported by the training artifacts.

## Fitted xT surface

The figure below is intended to show the **match-derived** xT grid, not a synthetic toy surface. The plotting code reads the fitted array directly; it does not retrain the model. Coordinates are displayed on a 105 × 68 m pitch, with the attacking direction left to right.

**Figure placeholder — Codex:** Verify the trained grid's orientation and axis convention against `xt_library.py` and `get_xt_values` before finalizing the plot. Replace the generic pitch rendering if the project has an established plotting helper.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Run from a location where the sibling analysis repository is accessible.
xt_path = Path('skillcorner-analysis/results/xt/xt_grid.npy')
if not xt_path.is_file():
    raise FileNotFoundError(
        f'{xt_path} not found. Set xt_path to the fitted grid produced by the xT training pipeline.'
    )
xt_grid = np.load(xt_path)
assert xt_grid.ndim == 2, 'Expected an (n_x, n_y) xT grid.'

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(
    xt_grid.T, origin='lower', extent=(-52.5, 52.5, -34, 34),
    aspect='equal', cmap='magma', interpolation='nearest'
)
# Pitch outline, halfway line, and centre circle (illustrative rendering).
from matplotlib.patches import Rectangle, Circle
ax.add_patch(Rectangle((-52.5, -34), 105, 68, fill=False, edgecolor='white', linewidth=1.2))
ax.plot([0, 0], [-34, 34], color='white', linewidth=1)
ax.add_patch(Circle((0, 0), 9.15, fill=False, edgecolor='white', linewidth=1))
for x0 in (-52.5, 36.0):
    ax.add_patch(Rectangle((x0, -20.16), 16.5, 40.32, fill=False, edgecolor='white', linewidth=1))
ax.set(xlabel='Pitch x (m) — attacking right', ylabel='Pitch y (m)', title='Fitted Expected Threat (xT)')
fig.colorbar(im, ax=ax, label='xT')
fig.tight_layout()
plt.show()


## Using xT in PRESS

PRESS expects an xT grid with shape `(n_x, n_y)` and uses centred pitch coordinates. The helper `get_xt_values` evaluates locations for both teams, accounting for attacking direction. This example uses the fitted grid loaded above.


In [ ]:
from PRESS.utils.state import get_xt_values

targets = np.array([[-35., -17.], [0., 17.], [35., -17.]])
team_xt, opponent_xt = get_xt_values(targets, 'left_to_right', xt_grid)
print('xT grid (n_x, n_y):', xt_grid.shape)
print('team xT:', team_xt)
print('opponent xT:', opponent_xt)


## References

Singh, K. (2018). *Introducing Expected Threat (xT).* [Project description](https://karun.in/blog/expected-threat.html). [@singh2018xt]
